# Controlled Gates — Amazon Braket

Controlled gates apply an operation to a target qubit conditioned on
a control qubit.  We explore CNOT (controlled-X) and CZ (controlled-Z).

In [ ]:
import json

from braket.circuit import Circuit
from braket.devices import LocalSimulator

In [ ]:
device = LocalSimulator()

## CNOT (CX) truth table

CNOT flips the target qubit when the control is |1\rangle.
Here qubit 1 is the control, qubit 0 is the target.

In [ ]:
print(f"  {'input':>8}  ->  {'output':>8}")
print(f"  {'--------':>8}     {'--------':>8}")

for control, target in ((0, 0), (0, 1), (1, 0), (1, 1)):
    circuit = Circuit()
    if target:
        circuit.x(0)
    if control:
        circuit.x(1)
    circuit.cnot(1, 0)  # control=q1, target=q0
    result = device.run(circuit, shots=0).result()
    amps = result.result_types[0].value
    out_idx = next(i for i, a in enumerate(amps) if abs(a) > 1e-10)
    out_bits = format(out_idx, '02b')
    print(f"  |{control}{target}>  ->  |{out_bits}>")

print()
print("CX flips qubit 0 when qubit 1 is |1>.")

## CZ truth table

CZ is diagonal: it only adds a phase of -1 to the |11\rangle component.

In [ ]:
for i in range(4):
    bits = format(i, '02b')
    circuit = Circuit()
    if bits[1] == '1':
        circuit.x(1)
    if bits[0] == '1':
        circuit.x(0)
    circuit.cz(1, 0)  # control=q1, target=q0
    result = device.run(circuit, shots=0).result()
    amps = result.result_types[0].value
    for idx, amp in enumerate(amps):
        if abs(amp) > 1e-10:
            basis = format(idx, '02b')
            print(f"  |{bits}> -> {amp:.4f} |{basis}>")

print()
print("CZ only modifies |11> — it adds a phase, not a bit flip.")

## CNOT with superposition input

H on the control creates the Bell state |\u03a6+\rangle.

In [ ]:
circuit = Circuit()
circuit.h(1)       # put control in superposition
circuit.cnot(1, 0) # CNOT
circuit.measure(0)
circuit.measure(1)
print(circuit)

result = device.run(circuit, shots=2000).result()
counts = result.result_types[0].value
print(f"counts: {counts}")
print("Expect |00> and |11> ~50/50 — the Bell state Phi+.")